# Advanced 09 lab — Operate a spatial AI service through drift, incident, and recovery

**Scenario.** A multi-camera inspection service supports classification, metric position, cross-camera fusion, and policy-document retrieval. Site A establishes validated baselines. Site B selects operational policy. Site C is reporting-only and introduces camera movement, index lag, delayed labels, and runtime tail regression.

**Authority boundary.** This deterministic notebook implements the contracts directly with common Python libraries. It does not connect to sensors, deploy services, modify production flags, or authorize a release. All decisions are teaching evidence with `authorization: none`.


## 1. Environment and disabled production mappings

The optional systems map the primitives to production components; they are deliberately not imported. A tool can persist or visualize evidence, but it cannot define the spatial contract on our behalf.


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from collections import defaultdict, deque
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.spatial.transform import Rotation
from scipy.spatial.distance import jensenshannon

SEED = 909
rng = np.random.default_rng(SEED)
COURSE_DIR = Path.cwd()
if COURSE_DIR.name != "09-production-spatial-ai-operations-observability":
    candidate = Path("curriculum/advanced/09-production-spatial-ai-operations-observability")
    if candidate.exists():
        COURSE_DIR = candidate
ARTIFACT_DIR = COURSE_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

OPTIONAL_SYSTEMS = pd.DataFrame([
    ("MLflow", False, "model/artifact registry mapping; does not cover spatial dependencies by itself"),
    ("OpenTelemetry", False, "trace/metric/log correlation mapping; custom domain semantics remain governed"),
    ("Prometheus + Grafana", False, "bounded metrics and dashboards; avoid unbounded labels"),
    ("ROS 2 tf2 + rosbag2", False, "time-aware transform/replay mapping; calibration validity remains explicit"),
    ("OpenLineage + SLSA", False, "data/build provenance mapping; capability evidence remains separate"),
    ("OpenFeature", False, "trusted flag/kill-switch mapping; provider integrity and authority still required"),
    ("Evidently / NannyML", False, "drift/delayed-label mapping under explicit assumptions"),
    ("Argo Rollouts / Flagger", False, "progressive delivery mapping; capability gates must be supplied"),
], columns=["system", "enabled", "boundary"])

environment_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "seed": SEED, "claim": "deterministic synthetic operations teaching evidence only",
}
pd.DataFrame([environment_manifest]), OPTIONAL_SYSTEMS


## 2. Typed production contracts

Registries have different semantics, but every referenced record is immutable. `PASS`, `FAIL`, and `MISSING` stay distinct. The model never promotes, rolls back, or changes a kill switch.


In [ ]:
Status = Literal["PASS", "FAIL", "MISSING"]
CapabilityState = Literal["VALID", "DEGRADED", "INVALID", "MISSING"]

@dataclass(frozen=True)
class RegistryRef:
    registry: str
    artifact_id: str
    revision: str
    digest: str

@dataclass(frozen=True)
class DeploymentManifest:
    deployment_id: str
    created_at: str
    model: RegistryRef
    processor: RegistryRef
    sensor_profile: RegistryRef
    calibration_bundle: RegistryRef
    frame_graph: RegistryRef
    retrieval_index: RegistryRef
    memory_schema: RegistryRef
    policy: RegistryRef
    runtime: RegistryRef
    hardware_profile: RegistryRef
    parent_deployment_id: str | None = None

@dataclass(frozen=True)
class CapabilitySLO:
    name: str
    direction: Literal["higher_is_better", "lower_is_better"]
    target: float
    population: str
    window: str
    minimum_labels: int
    evidence_class: Literal["immediate_proxy", "delayed_outcome"]

@dataclass(frozen=True)
class DecisionTrace:
    decision_id: str
    request_id: str
    observation_id: str
    deployment_id: str
    sensor_id: str
    observation_time: str
    prediction: str
    confidence: float
    policy_revision: str
    action: str

@dataclass(frozen=True)
class OutcomeTrace:
    decision_id: str
    outcome: str
    label_source: str
    observed_at: str

@dataclass(frozen=True)
class GateCheck:
    name: str
    status: Status
    detail: str

@dataclass(frozen=True)
class ControlDecision:
    operation: str
    outcome: Literal["PROMOTE", "HOLD", "REJECT", "ROLLBACK", "DISABLE_CAPABILITY", "RESTORE_CAPABILITY"]
    checks: tuple[GateCheck, ...]
    policy_hash: str
    actor: str = "trusted_control_plane_proxy"
    authorization: Literal["none"] = "none"

def canonical_hash(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def ref(registry, artifact_id, revision, payload):
    return RegistryRef(registry, artifact_id, revision, canonical_hash(payload))


## 3. Immutable registries and deployment manifest

The processor and model are separate identities. Calibration, frame graph, index, memory, policy, runtime, and hardware are first-class dependencies rather than free-form tags.


In [ ]:
MODEL_V1 = ref("model", "inspection-model", "1.4.0", {"architecture": "tiny-multitask", "training": "site-a"})
PROCESSOR_V1 = ref("processor", "inspection-processor", "2.1.0", {"resize": [640, 480], "normalization": "v3"})
SENSOR_A = ref("sensor", "camera-17", "serial-C17", {"resolution": [1920, 1080], "firmware": "4.2"})
CALIB_8 = ref("calibration", "camera-17-calibration", "008", {"fx": 912.4, "fy": 911.8, "valid_from": "2026-08-01T00:00:00Z"})
FRAMES_22 = ref("frame_graph", "line-a-frames", "22", {"map_frame": "world", "translation_unit": "metre"})
INDEX_91 = ref("retrieval_index", "manuals", "91", {"source_revision": 7, "encoder": "text-embed-4", "acl": "6"})
MEMORY_V3 = ref("memory_schema", "inspection-memory", "3", {"map_frame": "world", "reader": [2, 3]})
POLICY_14 = ref("policy", "inspection-policy", "14", {"metric_kill_on_invalid_calibration": True, "review_threshold": 0.62})
RUNTIME_8 = ref("runtime", "edge-runtime", "8", {"compiler": "reference", "precision": "fp32", "shape_policy": "static"})
HARDWARE_A = ref("hardware", "edge-a", "A3", {"accelerator": "cpu-proxy", "firmware": "11"})

manifest_v1 = DeploymentManifest(
    "inspection-prod-042", "2026-09-01T08:00:00Z", MODEL_V1, PROCESSOR_V1, SENSOR_A,
    CALIB_8, FRAMES_22, INDEX_91, MEMORY_V3, POLICY_14, RUNTIME_8, HARDWARE_A,
)
MANIFEST_V1_HASH = canonical_hash(asdict(manifest_v1))
registry_table = pd.DataFrame([asdict(value) for value in (
    MODEL_V1, PROCESSOR_V1, SENSOR_A, CALIB_8, FRAMES_22, INDEX_91, MEMORY_V3, POLICY_14, RUNTIME_8, HARDWARE_A
)])
assert len(set(registry_table.digest)) == len(registry_table)
pd.DataFrame([{"manifest_hash": MANIFEST_V1_HASH, **asdict(manifest_v1)}]).T.head(14), registry_table


## 4. Deterministic configuration compatibility

The checks compare declared semantics, not only tensor/vector shapes. A same-dimension encoder can still be incompatible with an index built from another representation.


In [ ]:
COMPATIBILITY = {
    "inspection-model@1.4.0": {"processor": "inspection-processor@2.1.0", "runtime": "edge-runtime@8", "memory_readers": {2, 3}},
    "inspection-model@1.3.0": {"processor": "inspection-processor@2.0.0", "runtime": "edge-runtime@7", "memory_readers": {2}},
}

def identity(record: RegistryRef) -> str:
    return f"{record.artifact_id}@{record.revision}"

def check_configuration_compatibility(manifest: DeploymentManifest, *, index_encoder="text-embed-4", memory_schema=3):
    declared = COMPATIBILITY.get(identity(manifest.model))
    checks = []
    checks.append(GateCheck("known_model_contract", "PASS" if declared else "MISSING", identity(manifest.model)))
    if declared:
        checks.append(GateCheck("processor_compatibility", "PASS" if identity(manifest.processor) == declared["processor"] else "FAIL", identity(manifest.processor)))
        checks.append(GateCheck("runtime_compatibility", "PASS" if identity(manifest.runtime) == declared["runtime"] else "FAIL", identity(manifest.runtime)))
        checks.append(GateCheck("memory_schema_readable", "PASS" if memory_schema in declared["memory_readers"] else "FAIL", f"schema={memory_schema}"))
    expected_encoder = "text-embed-4" if manifest.retrieval_index.revision == "91" else "unknown"
    checks.append(GateCheck("index_encoder_compatibility", "PASS" if index_encoder == expected_encoder else "FAIL", f"query={index_encoder}; index={expected_encoder}"))
    checks.append(GateCheck("calibration_sensor_binding", "PASS" if "camera-17" in manifest.calibration_bundle.artifact_id and manifest.sensor_profile.artifact_id == "camera-17" else "FAIL", manifest.calibration_bundle.artifact_id))
    return checks

compatibility_v1 = check_configuration_compatibility(manifest_v1)
same_dimension_wrong_semantics = check_configuration_compatibility(manifest_v1, index_encoder="other-encoder-same-dim")
assert all(check.status == "PASS" for check in compatibility_v1)
assert next(check for check in same_dimension_wrong_semantics if check.name == "index_encoder_compatibility").status == "FAIL"
pd.DataFrame([asdict(check) for check in compatibility_v1 + same_dimension_wrong_semantics])


## 5. Time-valid frame graph

Each edge stores a named direction, transform, unit, validity interval, and provenance. Historical observations use the path valid at capture time. `MISSING` is never converted to identity.


In [ ]:
@dataclass(frozen=True)
class TransformEdge:
    source: str
    target: str
    matrix: tuple[tuple[float, ...], ...]
    valid_from: str
    valid_until: str | None
    translation_unit: Literal["metre"]
    calibration_id: str

    def array(self):
        return np.asarray(self.matrix, dtype=float)

def T(x=0.0, y=0.0, z=0.0, yaw_deg=0.0):
    matrix = np.eye(4)
    matrix[:3, :3] = Rotation.from_euler("z", yaw_deg, degrees=True).as_matrix()
    matrix[:3, 3] = [x, y, z]
    return tuple(tuple(float(value) for value in row) for row in matrix)

EDGES = [
    TransformEdge("world", "robot_base", T(1.0, 0.0, 0.0), "2026-01-01T00:00:00Z", None, "metre", "survey-4"),
    TransformEdge("robot_base", "camera_mount", T(0.2, 0.0, 1.1), "2026-01-01T00:00:00Z", None, "metre", "mount-12"),
    TransformEdge("camera_mount", "camera_17", T(0.05, 0.0, 0.1), "2026-08-01T00:00:00Z", "2026-09-15T10:00:00Z", "metre", "calib-008"),
    TransformEdge("camera_mount", "camera_17", T(0.07, 0.01, 0.1, yaw_deg=1.2), "2026-09-15T10:00:00Z", None, "metre", "calib-009"),
]

def timestamp(text):
    return datetime.fromisoformat(text.replace("Z", "+00:00"))

def is_valid(edge: TransformEdge, at: str):
    value = timestamp(at)
    return timestamp(edge.valid_from) <= value and (edge.valid_until is None or value < timestamp(edge.valid_until))

def resolve_transform(source: str, target: str, at: str, edges=EDGES):
    graph = defaultdict(list)
    for edge in edges:
        if not is_valid(edge, at):
            continue
        graph[edge.source].append((edge.target, edge.array(), edge.calibration_id))
        graph[edge.target].append((edge.source, np.linalg.inv(edge.array()), edge.calibration_id))
    queue = deque([(source, np.eye(4), [])])
    visited = {source}
    while queue:
        frame, accumulated, lineage = queue.popleft()
        if frame == target:
            return {"status": "PASS", "transform": accumulated, "lineage": lineage}
        for neighbor, edge_matrix, calibration_id in graph[frame]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, edge_matrix @ accumulated, lineage + [calibration_id]))
    return {"status": "MISSING", "transform": None, "lineage": []}

historical = resolve_transform("world", "camera_17", "2026-09-10T12:00:00Z")
current = resolve_transform("world", "camera_17", "2026-09-18T12:00:00Z")
missing = resolve_transform("world", "unknown_camera", "2026-09-18T12:00:00Z")
assert historical["lineage"][-1] == "calib-008"
assert current["lineage"][-1] == "calib-009"
assert missing["status"] == "MISSING" and missing["transform"] is None
pd.DataFrame([
    {"query": "historical", "status": historical["status"], "x_m": historical["transform"][0, 3], "lineage": historical["lineage"]},
    {"query": "current", "status": current["status"], "x_m": current["transform"][0, 3], "lineage": current["lineage"]},
    {"query": "missing", "status": missing["status"], "x_m": None, "lineage": missing["lineage"]},
])


## 6. Cycle consistency and known-answer failure

A loop should return approximately to identity. The injected inconsistent edge has a measurable translation and rotation residual.


In [ ]:
def cycle_closure(matrices):
    composed = np.eye(4)
    for matrix in matrices:
        composed = np.asarray(matrix) @ composed
    translation_error_m = float(np.linalg.norm(composed[:3, 3]))
    rotation_error_deg = float(np.degrees(Rotation.from_matrix(composed[:3, :3]).magnitude()))
    return {"translation_closure_error_m": translation_error_m, "rotation_closure_error_deg": rotation_error_deg}

ab = np.asarray(T(1.0, 0.0, 0.0, 2.0))
bc = np.asarray(T(0.0, 1.0, 0.0, -2.0))
ca_consistent = np.linalg.inv(bc @ ab)
ca_inconsistent = np.asarray(T(-0.94, -1.03, 0.0, 1.5))
closure = pd.DataFrame([
    {"cycle": "consistent", **cycle_closure([ab, bc, ca_consistent])},
    {"cycle": "injected_mismatch", **cycle_closure([ab, bc, ca_inconsistent])},
])
assert closure.loc[closure.cycle == "consistent", "translation_closure_error_m"].iloc[0] < 1e-9
assert closure.loc[closure.cycle == "injected_mismatch", "translation_closure_error_m"].iloc[0] > 0.02
closure


## 7. Site A/B/C calibration residuals

Site A establishes healthy reference behavior. Site B selects persistence and thresholds. Site C is generated now but cannot change policy.


In [ ]:
def calibration_windows(site, medians, seed):
    local = np.random.default_rng(seed)
    rows = []
    for window, target_median in enumerate(medians):
        residuals = np.clip(local.normal(target_median, max(0.08, target_median * 0.18), 40), 0, None)
        rows.append({
            "site": site, "window": window, "samples": len(residuals),
            "median_px": float(np.median(residuals)), "p95_px": float(np.quantile(residuals, .95)),
            "max_px": float(residuals.max()),
        })
    return rows

calibration_evidence = pd.DataFrame(
    calibration_windows("Site A", [0.55] * 5, SEED + 1)
    + calibration_windows("Site B", [0.6, 0.7, 1.5, 1.7, 3.3, 3.6], SEED + 2)
    + calibration_windows("Site C", [0.7, 1.0, 2.5, 3.8, 4.2, 4.4], SEED + 3)
)

SITE_B_CALIBRATION_POLICY = {"warning_p95_px": 2.0, "invalid_p95_px": 4.0, "persistence_windows": 2, "minimum_landmarks": 30}

def classify_calibration_windows(frame, policy):
    warning_streak = invalid_streak = 0
    states = []
    for row in frame.sort_values("window").itertuples():
        if row.samples < policy["minimum_landmarks"]:
            state = "MISSING"
        else:
            warning_streak = warning_streak + 1 if row.p95_px >= policy["warning_p95_px"] else 0
            invalid_streak = invalid_streak + 1 if row.p95_px >= policy["invalid_p95_px"] else 0
            state = "invalid" if invalid_streak >= policy["persistence_windows"] else "warning" if warning_streak >= policy["persistence_windows"] else "healthy"
        states.append(state)
    result = frame.sort_values("window").copy()
    result["calibration_state"] = states
    return result

site_b_calibration = classify_calibration_windows(calibration_evidence.query("site == 'Site B'"), SITE_B_CALIBRATION_POLICY)
POLICY_HASH_BEFORE_SITE_C = canonical_hash(SITE_B_CALIBRATION_POLICY)
site_c_calibration = classify_calibration_windows(calibration_evidence.query("site == 'Site C'"), SITE_B_CALIBRATION_POLICY)
POLICY_HASH_AFTER_SITE_C = canonical_hash(SITE_B_CALIBRATION_POLICY)
assert POLICY_HASH_BEFORE_SITE_C == POLICY_HASH_AFTER_SITE_C
assert site_c_calibration.iloc[-1].calibration_state == "invalid"
site_b_calibration, site_c_calibration


## 8. Residual visualization and persistence

The state machine does not fail on one noisy window. Policy responds only after persistent evidence at sufficient sample size.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for site, frame in calibration_evidence.groupby("site"):
    ax.plot(frame.window, frame.p95_px, marker="o", label=site)
ax.axhline(SITE_B_CALIBRATION_POLICY["warning_p95_px"], color="#F59E42", linestyle="--", label="warning threshold")
ax.axhline(SITE_B_CALIBRATION_POLICY["invalid_p95_px"], color="#D9485F", linestyle="--", label="invalid threshold")
ax.set(xlabel="window", ylabel="p95 reprojection residual (px)", title="Calibration evidence by source")
ax.legend(ncol=2); ax.grid(alpha=.2); plt.tight_layout()


## 9. Capability dependency graph and selective invalidation

Calibration health is not model health. The control plane computes impact from dependencies and disables only affected paths.


In [ ]:
CAPABILITY_DEPENDENCIES = {
    "classification": {"sensor", "processor", "model", "runtime"},
    "metric_position": {"sensor", "calibration", "frame_graph", "geometry_model", "runtime"},
    "cross_camera_fusion": {"sensor", "calibration", "frame_graph", "association_policy", "runtime"},
    "manual_retrieval": {"retrieval_index", "retrieval_encoder", "acl_policy", "runtime"},
}

def capability_impact(component_states):
    rows = []
    for capability, dependencies in CAPABILITY_DEPENDENCIES.items():
        missing = sorted(dep for dep in dependencies if dep not in component_states)
        invalid = sorted(dep for dep in dependencies if component_states.get(dep) == "INVALID")
        degraded = sorted(dep for dep in dependencies if component_states.get(dep) == "DEGRADED")
        state = "MISSING" if missing else "INVALID" if invalid else "DEGRADED" if degraded else "VALID"
        rows.append({"capability": capability, "state": state, "missing": missing, "invalid": invalid, "degraded": degraded})
    return pd.DataFrame(rows)

healthy_components = {name: "VALID" for deps in CAPABILITY_DEPENDENCIES.values() for name in deps}
camera_moved_components = {**healthy_components, "calibration": "INVALID"}
camera_impact = capability_impact(camera_moved_components)
assert camera_impact.set_index("capability").loc["classification", "state"] == "VALID"
assert camera_impact.set_index("capability").loc["metric_position", "state"] == "INVALID"
assert camera_impact.set_index("capability").loc["cross_camera_fusion", "state"] == "INVALID"
camera_impact


## 10. Trusted selective kill switch

The model supplies no control-plane authority. The trusted proxy translates dependency evidence into audited capability state.


In [ ]:
def apply_selective_kill_switch(impact, policy_hash):
    checks = tuple(GateCheck(f"{row.capability}_dependency_state", "PASS" if row.state == "VALID" else "FAIL" if row.state == "INVALID" else "MISSING", row.state) for row in impact.itertuples())
    disabled = sorted(impact.loc[impact.state == "INVALID", "capability"])
    return {
        "disabled_capabilities": disabled,
        "remaining_capabilities": sorted(impact.loc[impact.state == "VALID", "capability"]),
        "decision": asdict(ControlDecision("calibration_incident_containment", "DISABLE_CAPABILITY", checks, policy_hash)),
    }

kill_switch_record = apply_selective_kill_switch(camera_impact, POLICY_HASH_BEFORE_SITE_C)
assert "metric_position" in kill_switch_record["disabled_capabilities"]
assert "classification" in kill_switch_record["remaining_capabilities"]
assert kill_switch_record["decision"]["actor"] == "trusted_control_plane_proxy"
kill_switch_record


## 11. Structured telemetry and cardinality

Metrics aggregate bounded dimensions. High-cardinality request, decision, and object identities remain in correlated traces/events rather than metric labels.


In [ ]:
telemetry_events = pd.DataFrame([
    {"event_type": "calibration_health", "deployment_id": manifest_v1.deployment_id, "sensor_id": "camera_17", "calibration_id": "calib-008", "window": int(row.window), "p95_reprojection_error_px": row.p95_px, "status": row.calibration_state}
    for row in site_c_calibration.itertuples()
])
safe_metric_labels = ["deployment_class", "site", "capability", "status"]
unsafe_metric_labels = ["request_id", "decision_id", "raw_object_id", "user_id", "request_text"]
cardinality_demo = pd.DataFrame([
    {"design": "bounded", "label_sets": 3 * 5 * 4 * 4, "examples": safe_metric_labels},
    {"design": "request_id_label", "label_sets": 1_000_000, "examples": unsafe_metric_labels},
])
assert cardinality_demo.loc[1, "label_sets"] > cardinality_demo.loc[0, "label_sets"] * 100
telemetry_events, cardinality_demo


## 12. Drift taxonomy with explicit windows

Each detector has one semantic target, a version, a reference/current window, and sample counts. Jensen–Shannon distance is a teaching signal here—not a universal drift score and not a capability decision.


In [ ]:
def histogram_js(reference, current, bins):
    ref_hist, _ = np.histogram(reference, bins=bins, density=False)
    cur_hist, _ = np.histogram(current, bins=bins, density=False)
    ref_prob = (ref_hist + 1e-9) / (ref_hist.sum() + 1e-9 * len(ref_hist))
    cur_prob = (cur_hist + 1e-9) / (cur_hist.sum() + 1e-9 * len(cur_hist))
    return float(jensenshannon(ref_prob, cur_prob, base=2.0))

drift_inputs = {
    "input_brightness": (rng.normal(.55, .08, 400), rng.normal(.48, .11, 400)),
    "focus_proxy": (rng.normal(.82, .04, 400), rng.normal(.66, .08, 400)),
    "embedding_norm": (rng.normal(1.0, .03, 400), rng.normal(.98, .04, 400)),
    "runtime_ms": (rng.lognormal(3.2, .16, 400), rng.lognormal(3.45, .27, 400)),
}
DRIFT_FAMILY = {"input_brightness": "data", "focus_proxy": "sensor", "embedding_norm": "representation", "runtime_ms": "runtime"}
drift_rows = []
for feature, (reference, current_values) in drift_inputs.items():
    bins = np.histogram_bin_edges(np.concatenate([reference, current_values]), bins=12)
    drift_rows.append({
        "family": DRIFT_FAMILY[feature], "feature": feature, "detector": "histogram_js_v1",
        "reference_window": "Site A validated baseline", "current_window": "Site C window 3",
        "reference_n": len(reference), "current_n": len(current_values),
        "distance": histogram_js(reference, current_values, bins), "meaning": "investigate; not automatic rollback",
    })
drift_table = pd.DataFrame(drift_rows)
drift_table


## 13. Stable decision and outcome traces

Decisions bind the manifest, sensor, observation time, policy, and action. Private chain-of-thought is neither required nor stored.


In [ ]:
base_time = datetime(2026, 9, 18, 12, 0, tzinfo=timezone.utc)
decisions = []
for index in range(80):
    observed = base_time + timedelta(seconds=15 * index)
    difficult = index % 7 == 0
    decisions.append(DecisionTrace(
        f"decision-{index:04d}", f"request-{index:04d}", f"observation-{index:04d}",
        manifest_v1.deployment_id, "camera_17", observed.isoformat(),
        "defect" if index % 5 == 0 else "normal", 0.58 if difficult else 0.91,
        POLICY_14.revision, "review" if difficult else "accept",
    ))
decision_frame = pd.DataFrame([asdict(item) for item in decisions])

# Deliberately biased labels: reviewers select difficult/reviewed cases more often.
outcomes = []
for index, decision in enumerate(decisions):
    selected = decision.action == "review" or index % 9 == 0
    if selected:
        delay_hours = 2 + (index % 5) * 8
        truth = "defect" if index % 6 == 0 else "normal"
        outcomes.append(OutcomeTrace(decision.decision_id, truth, "human_review", (timestamp(decision.observation_time) + timedelta(hours=delay_hours)).isoformat()))
outcomes.append(outcomes[0])  # intentional duplicate delivery
outcome_frame = pd.DataFrame([asdict(item) for item in outcomes])
decision_frame.head(), outcome_frame.head()


## 14. Delayed outcome join, coverage, delay, and duplicate policy

`decision_id` is the durable join key. Duplicate outcomes are reported and deterministically de-duplicated; they are not silently counted twice.


In [ ]:
def join_decisions_to_outcomes(decision_rows, outcome_rows):
    duplicates = outcome_rows.duplicated("decision_id", keep=False)
    duplicate_ids = sorted(outcome_rows.loc[duplicates, "decision_id"].unique())
    unique_outcomes = outcome_rows.sort_values("observed_at").drop_duplicates("decision_id", keep="last")
    joined = decision_rows.merge(unique_outcomes, on="decision_id", how="left", validate="one_to_one")
    joined["observation_time"] = pd.to_datetime(joined["observation_time"], utc=True)
    joined["observed_at"] = pd.to_datetime(joined["observed_at"], utc=True)
    joined["label_delay_hours"] = (joined["observed_at"] - joined["observation_time"]).dt.total_seconds() / 3600
    labeled = joined["outcome"].notna()
    summary = {
        "decisions": len(joined), "unique_labeled": int(labeled.sum()),
        "label_coverage": float(labeled.mean()), "unmatched_decisions": int((~labeled).sum()),
        "duplicate_outcome_ids": duplicate_ids,
        "median_label_delay_hours": float(joined.loc[labeled, "label_delay_hours"].median()),
        "review_share_labeled": float((joined.loc[labeled, "action"] == "review").mean()),
        "review_share_population": float((joined["action"] == "review").mean()),
    }
    return joined, summary

joined_outcomes, outcome_join_summary = join_decisions_to_outcomes(decision_frame, outcome_frame)
assert outcome_join_summary["duplicate_outcome_ids"] == ["decision-0000"]
assert outcome_join_summary["review_share_labeled"] > outcome_join_summary["review_share_population"]
pd.DataFrame([outcome_join_summary])


## 15. Why timestamp joins are unsafe

Two close decisions can share a coarse timestamp or arrive out of order. Stable IDs preserve identity; nearest-time matching creates plausible but unauditable associations.


In [ ]:
timestamp_counterexample = pd.DataFrame([
    {"decision_id": "d-a", "coarse_time": "12:00:00", "prediction": "normal"},
    {"decision_id": "d-b", "coarse_time": "12:00:00", "prediction": "defect"},
]).merge(pd.DataFrame([
    {"decision_id": "d-b", "coarse_time": "12:00:00", "outcome": "defect"}
]), on="coarse_time", how="left", suffixes=("_decision", "_outcome"))
assert len(timestamp_counterexample) == 2  # one outcome ambiguously attaches to two decisions
timestamp_counterexample


## 16. Capability SLO windows and tri-state evaluation

A small labeled subset is `MISSING`, not a pass. Direction, population, minimum labels, and evidence class travel with the threshold.


In [ ]:
SLOS = [
    CapabilitySLO("small_defect_recall", "higher_is_better", .95, "inspection_line_A", "7d", 30, "delayed_outcome"),
    CapabilitySLO("metric_position_p95_cm", "lower_is_better", 1.5, "inspection_line_A", "1h", 30, "immediate_proxy"),
    CapabilitySLO("retrieval_complete_evidence_recall", "higher_is_better", .92, "current_policy_queries", "24h", 20, "delayed_outcome"),
]

def evaluate_slo(slo, value, sample_count):
    if value is None or sample_count < slo.minimum_labels:
        return GateCheck(slo.name, "MISSING", f"value={value}; n={sample_count}; minimum={slo.minimum_labels}")
    passes = value >= slo.target if slo.direction == "higher_is_better" else value <= slo.target
    return GateCheck(slo.name, "PASS" if passes else "FAIL", f"value={value:.3f}; target={slo.target:.3f}; n={sample_count}")

site_b_slo_checks = [
    evaluate_slo(SLOS[0], .96, 120), evaluate_slo(SLOS[1], 1.1, 160), evaluate_slo(SLOS[2], .94, 80),
]
site_c_slo_checks = [
    evaluate_slo(SLOS[0], .91, 18), evaluate_slo(SLOS[1], 4.8, 160), evaluate_slo(SLOS[2], .88, 14),
]
assert site_c_slo_checks[0].status == "MISSING"
assert site_c_slo_checks[1].status == "FAIL"
pd.DataFrame([asdict(check) for check in site_b_slo_checks + site_c_slo_checks])


## 17. Shadow and canary promotion with label delay

An otherwise healthy canary is held when delayed capability evidence is incomplete. Proxy health does not silently replace the required evidence class.


In [ ]:
def promotion_decision(checks, policy_hash):
    statuses = {check.status for check in checks}
    outcome = "REJECT" if "FAIL" in statuses else "HOLD" if "MISSING" in statuses else "PROMOTE"
    return ControlDecision("canary_promotion", outcome, tuple(checks), policy_hash)

canary_checks = [
    GateCheck("availability", "PASS", "0.9997"),
    GateCheck("p95_latency", "PASS", "74 ms <= 90 ms"),
    GateCheck("calibration_health", "PASS", "healthy"),
    GateCheck("small_defect_recall_24h", "MISSING", "labels arrive after 24h; only 6 observed"),
]
canary_decision = promotion_decision(canary_checks, POLICY_HASH_BEFORE_SITE_C)
assert canary_decision.outcome == "HOLD"
assert canary_decision.authorization == "none"
pd.DataFrame([asdict(check) for check in canary_decision.checks]), asdict(canary_decision)


## 18. Rollback bundle and state compatibility

Model-only rollback fails because the old model expects an older processor/runtime and cannot read memory schema v3. A complete bundle chooses a compatible snapshot and invalidates lineage-bound caches.


In [ ]:
MODEL_V0 = ref("model", "inspection-model", "1.3.0", {"architecture": "tiny-multitask", "training": "site-a-old"})
PROCESSOR_V0 = ref("processor", "inspection-processor", "2.0.0", {"resize": [640, 480], "normalization": "v2"})
RUNTIME_V0 = ref("runtime", "edge-runtime", "7", {"compiler": "reference", "precision": "fp32"})
MEMORY_V2 = ref("memory_schema", "inspection-memory", "2", {"map_frame": "world", "reader": [2]})

model_only_rollback = DeploymentManifest(
    "rollback-broken", "2026-09-18T14:00:00Z", MODEL_V0, PROCESSOR_V1, SENSOR_A,
    CALIB_8, FRAMES_22, INDEX_91, MEMORY_V3, POLICY_14, RUNTIME_8, HARDWARE_A, manifest_v1.deployment_id,
)
complete_rollback = DeploymentManifest(
    "rollback-complete", "2026-09-18T14:05:00Z", MODEL_V0, PROCESSOR_V0, SENSOR_A,
    CALIB_8, FRAMES_22, INDEX_91, MEMORY_V2, POLICY_14, RUNTIME_V0, HARDWARE_A, manifest_v1.deployment_id,
)
broken_checks = check_configuration_compatibility(model_only_rollback, memory_schema=3)
complete_checks = check_configuration_compatibility(complete_rollback, memory_schema=2)
assert any(check.status == "FAIL" for check in broken_checks)
assert all(check.status == "PASS" for check in complete_checks)
rollback_comparison = pd.DataFrame([
    {"bundle": "model_only", **{check.name: check.status for check in broken_checks}, "cache_action": "undefined"},
    {"bundle": "complete", **{check.name: check.status for check in complete_checks}, "cache_action": "invalidate model/processor/runtime lineage"},
])
rollback_comparison


## 19. Forward-only side effects

Rollback changes future routing and decisions. It cannot unsend a ticket, undo an executed physical action, or erase a human decision. Compensation and audit are separate workflows.


In [ ]:
side_effects = pd.DataFrame([
    ("embedding_cache", True, "invalidate by lineage"),
    ("memory_snapshot", True, "restore only after compatibility and loss review"),
    ("external_ticket_sent", False, "append correction/closure; preserve history"),
    ("physical_action_executed", False, "make safe, investigate, record consequence"),
    ("human_decision_completed", False, "notify reviewer and preserve audit trail"),
], columns=["side_effect", "rollback_reversible", "response"])
assert not side_effects.query("side_effect == 'physical_action_executed'").rollback_reversible.iloc[0]
side_effects


## 20. Incident timeline and earliest failure attribution

Per-incident durations are reported. One synthetic incident cannot justify a meaningful mean MTTD or MTTR.


In [ ]:
incident_start = timestamp("2026-09-18T12:30:00Z")
incident_timeline = pd.DataFrame([
    ("camera_bumped", incident_start, "trigger"),
    ("first_bad_metric_decision", incident_start + timedelta(seconds=18), "impact"),
    ("persistent_calibration_signal", incident_start + timedelta(minutes=3), "first_detectable"),
    ("alert", incident_start + timedelta(minutes=4), "detection"),
    ("acknowledged", incident_start + timedelta(minutes=7), "response"),
    ("metric_capability_disabled", incident_start + timedelta(minutes=9), "containment"),
    ("recalibration_registered", incident_start + timedelta(minutes=31), "mitigation"),
    ("independent_probe_passed", incident_start + timedelta(minutes=38), "recovery"),
    ("closed", incident_start + timedelta(minutes=52), "closure"),
], columns=["event", "timestamp", "phase"])
first_bad = incident_timeline.query("event == 'first_bad_metric_decision'").timestamp.iloc[0]
alert_time = incident_timeline.query("event == 'alert'").timestamp.iloc[0]
recovery_time = incident_timeline.query("event == 'independent_probe_passed'").timestamp.iloc[0]
incident_durations = {
    "time_to_detect_minutes": (alert_time - first_bad).total_seconds() / 60,
    "time_to_recover_minutes": (recovery_time - first_bad).total_seconds() / 60,
    "reporting_note": "per-incident duration; not a statistically meaningful mean",
}
incident_timeline, pd.DataFrame([incident_durations])


## 21. Incident taxonomy and runbooks

Different incidents require different containment. Model rollback is not the default response to every symptom.


In [ ]:
incident_catalog = pd.DataFrame([
    ("camera_movement", "reprojection residual persistence", "metric_position; cross_camera_fusion", "kill metric capability; recalibrate", "landmark + frame probes"),
    ("stale_index", "source/index revision mismatch", "current_policy_retrieval", "disable current-only answers; rebuild index", "source/index/ACL parity"),
    ("runtime_regression", "deployment-bound p99 + queue + freshness", "all deadline-bound capabilities", "restore runtime bundle; shed load", "load + numeric + capability probes"),
    ("model_or_source_shift", "delayed capability regression", "task-specific", "verify labels/sensor/processor/calibration before adaptation", "source-held-out capability suite"),
], columns=["incident_class", "trigger", "affected_capabilities", "containment", "recovery_verification"])
incident_catalog


## 22. Recalibration recovery creates a new deployment

Recovery registers calibration `009`, creates a new immutable manifest, and remains disabled until independent landmark and frame-graph probes pass.


In [ ]:
CALIB_9 = ref("calibration", "camera-17-calibration", "009", {"fx": 912.1, "fy": 911.5, "valid_from": "2026-09-18T13:01:00Z", "method": "post-movement-recalibration"})
recovered_manifest = DeploymentManifest(
    "inspection-prod-043", "2026-09-18T13:08:00Z", MODEL_V1, PROCESSOR_V1, SENSOR_A,
    CALIB_9, FRAMES_22, INDEX_91, MEMORY_V3, POLICY_14, RUNTIME_8, HARDWARE_A, manifest_v1.deployment_id,
)
recovery_checks = (
    GateCheck("configuration_compatibility", "PASS", "all refs and reader contracts compatible"),
    GateCheck("landmark_p95", "PASS", "0.91 px <= 2.0 px warning threshold"),
    GateCheck("frame_cycle_translation", "PASS", "0.003 m <= 0.01 m"),
    GateCheck("metric_position_probe", "PASS", "p95 0.9 cm <= 1.5 cm"),
    GateCheck("classification_regression", "PASS", "unchanged within declared tolerance"),
)
recovery_decision = ControlDecision("restore_metric_capability", "RESTORE_CAPABILITY", recovery_checks, POLICY_HASH_BEFORE_SITE_C)
assert recovered_manifest.deployment_id != manifest_v1.deployment_id
assert recovered_manifest.calibration_bundle.digest != manifest_v1.calibration_bundle.digest
assert all(check.status == "PASS" for check in recovery_decision.checks)
asdict(recovery_decision)


## 23. Stale-index and runtime incidents remain distinct

The stale index can return a fluent, well-cited answer to the wrong revision. The runtime can preserve numeric outputs while missing freshness/deadline contracts.


In [ ]:
secondary_incidents = pd.DataFrame([
    {"incident": "stale_index", "model_numeric_parity": True, "source_revision": 8, "index_revision": 7, "p99_ms": 74, "capability": "current_policy_retrieval", "status": "INVALID"},
    {"incident": "runtime_regression", "model_numeric_parity": True, "source_revision": 8, "index_revision": 8, "p99_ms": 184, "capability": "fresh_inspection_decision", "status": "INVALID"},
])
assert secondary_incidents.model_numeric_parity.all()
assert (secondary_incidents.status == "INVALID").all()
secondary_incidents


## 24. Site C frozen-policy report

No Site-C evidence changes calibration thresholds, SLOs, minimum samples, canary rules, or recovery requirements. The report distinguishes confirmed failure, missing outcome evidence, and healthy unaffected capability.


In [ ]:
site_c_report = pd.DataFrame([
    {"capability": "classification", "component_state": "VALID", "slo_status": "MISSING", "operating_action": "continue with existing reliability policy", "evidence": "processor/model/runtime healthy; delayed task labels incomplete"},
    {"capability": "metric_position", "component_state": "INVALID", "slo_status": "FAIL", "operating_action": "kill capability; recalibrate; independently verify", "evidence": "persistent calibration residual and position error"},
    {"capability": "cross_camera_fusion", "component_state": "INVALID", "slo_status": "MISSING", "operating_action": "kill capability", "evidence": "depends on invalid extrinsics"},
    {"capability": "current_policy_retrieval", "component_state": "INVALID", "slo_status": "MISSING", "operating_action": "disable current-only answers; rebuild", "evidence": "source/index revision lag"},
])
FINAL_POLICY_HASH = canonical_hash(SITE_B_CALIBRATION_POLICY)
assert FINAL_POLICY_HASH == POLICY_HASH_BEFORE_SITE_C
site_c_report


## 25. Incident evidence pack and postmortem

The pack uses digests and compact evidence rather than raw imagery. The camera bump is the trigger; the systemic root cause is failure to gate metric decisions on calibration health.


In [ ]:
postmortem = {
    "incident_id": "INC-2026-0918-CAMERA17",
    "impact": "metric position and cross-camera fusion invalid for 36 minutes; classification remained available",
    "trigger": "camera_17 physically moved",
    "root_cause": "metric capability initially lacked a persistence-aware calibration dependency gate",
    "contributing_factors": ["transform health was not linked to capability state", "label-aware geometry outcome arrived later"],
    "what_worked": ["structured deployment-bound telemetry", "selective kill switch", "known landmark fixture"],
    "what_failed": ["initial deployment permitted metric decisions during warning persistence"],
    "corrective_actions": [
        {"action": "enforce calibration dependency gate", "owner": "spatial-platform", "due": "2026-09-30", "verification": "quarterly camera-movement drill"},
        {"action": "add transform-cycle alert", "owner": "robotics-platform", "due": "2026-10-07", "verification": "fault-injection test"},
    ],
}

evidence_pack = {
    "course": "Advanced 09 — Production Spatial AI Operations & Observability",
    "incident_id": postmortem["incident_id"],
    "deployment_manifest_before": {"deployment_id": manifest_v1.deployment_id, "digest": MANIFEST_V1_HASH},
    "deployment_manifest_after": {"deployment_id": recovered_manifest.deployment_id, "digest": canonical_hash(asdict(recovered_manifest))},
    "policy_hash_before_site_c": POLICY_HASH_BEFORE_SITE_C,
    "policy_hash_after_site_c": POLICY_HASH_AFTER_SITE_C,
    "compatibility_checks": [asdict(check) for check in compatibility_v1],
    "calibration_policy": SITE_B_CALIBRATION_POLICY,
    "calibration_site_c": site_c_calibration.to_dict(orient="records"),
    "capability_impact": camera_impact.to_dict(orient="records"),
    "kill_switch": kill_switch_record,
    "outcome_join": outcome_join_summary,
    "canary_decision": asdict(canary_decision),
    "rollback": {"model_only": [asdict(check) for check in broken_checks], "complete_bundle": [asdict(check) for check in complete_checks]},
    "incident_durations": incident_durations,
    "recovery_decision": asdict(recovery_decision),
    "site_c_report": site_c_report.to_dict(orient="records"),
    "postmortem": postmortem,
    "privacy": {"raw_images_stored": False, "private_reasoning_stored": False, "decision_ids_in_metrics": False},
    "site_c_role": "reporting_only_no_changes",
    "authorization": "none",
}
assert evidence_pack["policy_hash_before_site_c"] == evidence_pack["policy_hash_after_site_c"]
assert evidence_pack["authorization"] == "none"

(ARTIFACT_DIR / "production_spatial_ai_evidence.json").write_text(json.dumps(evidence_pack, indent=2, default=str) + "\n", encoding="utf-8")
incident_timeline.assign(timestamp=incident_timeline.timestamp.astype(str)).to_csv(ARTIFACT_DIR / "production_spatial_ai_incident_timeline.csv", index=False)
telemetry_events.to_csv(ARTIFACT_DIR / "production_spatial_ai_telemetry.csv", index=False)
{
    "evidence": str(ARTIFACT_DIR / "production_spatial_ai_evidence.json"),
    "timeline": str(ARTIFACT_DIR / "production_spatial_ai_incident_timeline.csv"),
    "telemetry": str(ARTIFACT_DIR / "production_spatial_ai_telemetry.csv"),
    "authorization": evidence_pack["authorization"],
}


## 26. Enterprise decision artifact

The final decision is not “deploy the model.” It states which capabilities may operate, which remain disabled, why, which evidence is missing, and what a trusted operator must verify next.


In [ ]:
enterprise_decision = {
    "deployment": recovered_manifest.deployment_id,
    "restore": ["metric_position", "classification"],
    "remain_disabled": ["current_policy_retrieval until index revision 8 verification"],
    "hold": ["canary promotion until minimum delayed labels arrive"],
    "required_follow_up": ["index rebuild parity", "24h capability window", "corrective-action owners acknowledge"],
    "decision_owner": "trusted operations control plane",
    "authorization": "none",
}
assert enterprise_decision["authorization"] == "none"
enterprise_decision


## 27. Final checks

Before adapting this pattern, verify that you can explain why model health, calibration health, frame-graph health, retrieval freshness, runtime health, and outcome evidence are distinct—and why recovery remains incomplete until the affected capability passes an independent probe.


In [ ]:
final_checks = {
    "immutable_manifest": MANIFEST_V1_HASH != canonical_hash(asdict(recovered_manifest)),
    "historical_transform_uses_historical_calibration": historical["lineage"][-1] == "calib-008",
    "missing_transform_not_identity": missing["transform"] is None,
    "selective_degradation": "classification" in kill_switch_record["remaining_capabilities"],
    "site_c_policy_frozen": POLICY_HASH_BEFORE_SITE_C == FINAL_POLICY_HASH,
    "canary_missing_evidence_holds": canary_decision.outcome == "HOLD",
    "model_only_rollback_rejected": any(check.status == "FAIL" for check in broken_checks),
    "complete_rollback_compatible": all(check.status == "PASS" for check in complete_checks),
    "recovery_independently_verified": all(check.status == "PASS" for check in recovery_decision.checks),
    "non_authorizing": evidence_pack["authorization"] == "none",
}
assert all(final_checks.values())
pd.DataFrame(final_checks.items(), columns=["check", "passed"])
